# Multi-Region LLM Agent Infrastructure — Experiment Analysis
IEEE TKDE DK-GenAI Special Issue

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from pathlib import Path

sns.set_theme(style='whitegrid', font_scale=1.2)
RESULTS_DIR = Path('../results')
FIGURES_DIR = Path('../figures')
FIGURES_DIR.mkdir(exist_ok=True)

## Load Experiment Results

In [ ]:
exp_a = pd.read_csv(RESULTS_DIR / 'experiment_a/experiment_a_raw.csv')
exp_b = pd.read_csv(RESULTS_DIR / 'experiment_b/experiment_b_raw.csv')
exp_c = pd.read_csv(RESULTS_DIR / 'experiment_c/experiment_c_raw.csv')

print('Exp A:', exp_a.shape)
print('Exp B:', exp_b.shape)
print('Exp C:', exp_c.shape)

## Figure 1 — Experiment A: Write Engine Latency CDF

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

WRITE_ALGO_LABELS = {
    'W0': 'W0: Baseline (naive)',
    'W1': 'W1: Selective Flush',
    'W2': 'W2: WAL + Async Batch',
    'W3': 'W3: CRDT Merge',
    'W4': 'W4: Adaptive Pre-flush',
}

colors = sns.color_palette('tab10', n_colors=5)

for ax, metric, title in zip(
    axes,
    ['write_latency_ms', 'flush_latency_ms'],
    ['Per-trace Write Latency (ms)', 'Flush Latency (ms)'],
):
    for i, algo in enumerate(['W0', 'W1', 'W2', 'W3', 'W4']):
        subset = exp_a[exp_a['write_algorithm'] == algo][metric].dropna()
        if subset.empty:
            continue
        sorted_vals = np.sort(subset)
        cdf = np.arange(1, len(sorted_vals) + 1) / len(sorted_vals)
        ax.plot(sorted_vals, cdf, label=WRITE_ALGO_LABELS[algo], color=colors[i], lw=2)
    ax.set_xlabel(title)
    ax.set_ylabel('CDF')
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.set_xlim(left=0)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig1_write_engine_cdf.pdf', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig1_write_engine_cdf.png', dpi=150, bbox_inches='tight')
plt.show()

## Figure 2 — Experiment A: Write Latency Box Plot (p50/p95/p99)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
order = ['W0', 'W1', 'W2', 'W3', 'W4']
sns.boxplot(
    data=exp_a,
    x='write_algorithm', y='write_latency_ms',
    order=order, palette='tab10', ax=ax,
    showfliers=False,
)
ax.set_xlabel('Write Algorithm')
ax.set_ylabel('Write Latency (ms)')
ax.set_title('Experiment A: Write Engine Comparison')
ax.set_xticklabels([WRITE_ALGO_LABELS.get(a, a) for a in order], rotation=15, ha='right')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig2_write_boxplot.pdf', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig2_write_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

## Figure 3 — Experiment B: Read Engine Handoff Latency

In [ ]:
READ_ALGO_LABELS = {
    'R0': 'R0: Full Dump (baseline)',
    'R1': 'R1: Hydration Protocol',
    'R2': 'R2: LLM Summarization',
    'R3': 'R3: Semantic RAG',
    'R4': 'R4: MemGPT Hierarchical',
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics_titles = [
    ('handoff_latency_ms', 'Handoff Latency (ms)'),
    ('context_token_count', 'Context Tokens (input)'),
    ('state_integrity_score', 'State Integrity Score'),
]

order_r = ['R0', 'R1', 'R2', 'R3', 'R4']
for ax, (metric, title) in zip(axes, metrics_titles):
    sns.boxplot(
        data=exp_b, x='read_algorithm', y=metric,
        order=order_r, palette='tab10', ax=ax, showfliers=False,
    )
    ax.set_xlabel('Read Algorithm')
    ax.set_ylabel(title)
    ax.set_title(title)
    ax.set_xticklabels([READ_ALGO_LABELS.get(a, a) for a in order_r], rotation=20, ha='right', fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig3_read_engine_comparison.pdf', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig3_read_engine_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Figure 4 — Experiment B: Compression Ratio vs Integrity Trade-off

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
colors_r = sns.color_palette('tab10', 5)

for i, algo in enumerate(['R0', 'R1', 'R2', 'R3', 'R4']):
    subset = exp_b[exp_b['read_algorithm'] == algo]
    ax.scatter(
        subset['compression_ratio'],
        subset['state_integrity_score'],
        label=READ_ALGO_LABELS.get(algo, algo),
        alpha=0.5, color=colors_r[i], s=30,
    )

ax.set_xlabel('Compression Ratio (higher = smaller payload)')
ax.set_ylabel('State Integrity Score')
ax.set_title('Read Engine: Compression vs Integrity Trade-off')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig4_compression_vs_integrity.pdf', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig4_compression_vs_integrity.png', dpi=150, bbox_inches='tight')
plt.show()

## Figure 5 — Experiment C: Hybrid vs All Conditions (Grouped Bar)

In [ ]:
summary_c = exp_c.groupby('condition').agg(
    write_p50=('write_latency_ms', lambda x: np.percentile(x, 50)),
    handoff_p50=('handoff_latency_ms', lambda x: np.percentile(x, 50)),
    cost_mean=('estimated_cost_usd', 'mean'),
    tokens_mean=('context_token_count', 'mean'),
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
conditions = summary_c['condition'].tolist()
x = np.arange(len(conditions))
w = 0.35

axes[0].bar(x, summary_c['write_p50'], width=w, label='Write p50 (ms)', color='steelblue')
axes[0].bar(x + w, summary_c['handoff_p50'], width=w, label='Handoff p50 (ms)', color='coral')
axes[0].set_xticks(x + w / 2)
axes[0].set_xticklabels(conditions, rotation=20, ha='right', fontsize=9)
axes[0].set_ylabel('Latency (ms)')
axes[0].set_title('Experiment C: Latency by Condition')
axes[0].legend()

axes[1].bar(x, summary_c['cost_mean'] * 1000, width=0.6, color='mediumseagreen')
axes[1].set_xticks(x)
axes[1].set_xticklabels(conditions, rotation=20, ha='right', fontsize=9)
axes[1].set_ylabel('Avg Cost per Iteration (millicents)')
axes[1].set_title('Experiment C: Cost by Condition')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig5_hybrid_comparison.pdf', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig5_hybrid_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Figure 6 — Cost Scaling: Cumulative API Cost vs Step Sequence

Maps to paper metric: `step_sequence_number` (X) vs `cumulative_cost_usd` (Y).
Shows how total spend grows with experiment progress for each condition —
optimized algorithms diverge visibly from the W0/R0 baseline slope.

In [ ]:
all_data = pd.concat([exp_a, exp_b, exp_c], ignore_index=True)

# Sort by global step and compute per-condition cumulative cost
scaling = all_data.sort_values('step_sequence_number').copy()
conditions_all = scaling['condition'].unique()
palette_all = sns.color_palette('tab20', n_colors=len(conditions_all))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: cumulative cost
for i, cond in enumerate(sorted(conditions_all)):
    sub = scaling[scaling['condition'] == cond].sort_values('step_sequence_number')
    sub = sub.assign(cumcost=sub['estimated_cost_usd'].cumsum())
    axes[0].plot(
        sub['step_sequence_number'], sub['cumcost'],
        label=cond, color=palette_all[i], lw=1.8,
    )
axes[0].set_xlabel('Step Sequence Number')
axes[0].set_ylabel('Cumulative Cost (USD)')
axes[0].set_title('API Cost Scaling by Condition')
axes[0].legend(fontsize=7, ncol=2)
axes[0].yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.4f'))

# Right: cumulative input tokens
for i, cond in enumerate(sorted(conditions_all)):
    sub = scaling[scaling['condition'] == cond].sort_values('step_sequence_number')
    sub = sub.assign(cumtok=sub['input_tokens_used'].cumsum())
    axes[1].plot(
        sub['step_sequence_number'], sub['cumtok'],
        label=cond, color=palette_all[i], lw=1.8,
    )
axes[1].set_xlabel('Step Sequence Number')
axes[1].set_ylabel('Cumulative Input Tokens')
axes[1].set_title('Token Consumption Scaling by Condition')
axes[1].legend(fontsize=7, ncol=2)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig6_cost_scaling.pdf', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig6_cost_scaling.png', dpi=150, bbox_inches='tight')
plt.show()

total_cost = all_data['estimated_cost_usd'].sum()
print(f'Total experiment cost: ${total_cost:.4f}')
print(f'Total iterations: {len(all_data)}')
print(f'Avg cost/iter: ${total_cost / max(len(all_data), 1):.6f}')

## Figure 7 — Latency Survival: Execution Latency vs WAN Latency

Maps to paper metric: `simulated_wan_latency_ms` (X) vs `execution_latency_ms` (Y).
When Toxiproxy is active, WAN latency varies per iteration — this plot reveals
which algorithms are most sensitive to cross-region network conditions.

**Note:** If `wan_simulation_active = False` for all rows (Toxiproxy not running),
WAN latency will be 0 for all; the right panel falls back to a CDF of execution latency.

In [ ]:
latency_df = all_data[[
    'condition', 'write_algorithm', 'read_algorithm',
    'simulated_wan_latency_ms', 'execution_latency_ms',
    'handoff_latency_ms', 'write_latency_ms',
    'wan_simulation_active',
]].copy()

wan_active = latency_df['wan_simulation_active'].any()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: scatter WAN RTT vs execution latency (useful when Toxiproxy is on)
read_algos = sorted(latency_df['read_algorithm'].unique())
palette_r = sns.color_palette('tab10', n_colors=len(read_algos))

for i, algo in enumerate(read_algos):
    sub = latency_df[latency_df['read_algorithm'] == algo]
    axes[0].scatter(
        sub['simulated_wan_latency_ms'],
        sub['execution_latency_ms'],
        label=READ_ALGO_LABELS.get(algo, algo),
        alpha=0.4, color=palette_r[i], s=20,
    )

axes[0].set_xlabel('Simulated WAN RTT (ms)')
axes[0].set_ylabel('Execution Latency (ms)')
axes[0].set_title(
    'WAN RTT vs Execution Latency'
    + (' [Toxiproxy active]' if wan_active else ' [no WAN simulation]')
)
axes[0].legend(fontsize=8)

# Right: execution latency CDF per read algorithm
for i, algo in enumerate(read_algos):
    sub = latency_df[latency_df['read_algorithm'] == algo]['execution_latency_ms'].dropna()
    if sub.empty:
        continue
    sv = np.sort(sub)
    cdf = np.arange(1, len(sv) + 1) / len(sv)
    axes[1].plot(sv, cdf, label=READ_ALGO_LABELS.get(algo, algo), color=palette_r[i], lw=2)

axes[1].set_xlabel('Execution Latency (ms)')
axes[1].set_ylabel('CDF')
axes[1].set_title('Execution Latency CDF by Read Algorithm')
axes[1].legend(fontsize=8)
axes[1].set_xlim(left=0)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig7_latency_survival.pdf', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig7_latency_survival.png', dpi=150, bbox_inches='tight')
plt.show()

print('WAN simulation active in any row:', wan_active)
print(latency_df.groupby('read_algorithm')[['execution_latency_ms', 'simulated_wan_latency_ms']].describe().round(2))

## Figure 8 — Attention / Context Fidelity Tracking

Maps to paper metric: `step_sequence_number` (X) vs `retrieval_accuracy_score` (Y).
Shows how well each read algorithm preserves milestone content across the
experiment run. Rolling average (window=10) smooths per-iteration noise.

In [ ]:
attn_df = all_data[[
    'step_sequence_number', 'condition', 'read_algorithm',
    'retrieval_accuracy_score', 'state_integrity_score',
    'input_tokens_used', 'compression_ratio',
]].sort_values('step_sequence_number').copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ROLL_WINDOW = 10

# Left: retrieval_accuracy_score over steps, per read algorithm
read_algos_attn = sorted(attn_df['read_algorithm'].unique())
palette_attn = sns.color_palette('tab10', n_colors=len(read_algos_attn))

for i, algo in enumerate(read_algos_attn):
    sub = attn_df[attn_df['read_algorithm'] == algo].sort_values('step_sequence_number')
    rolled = sub['retrieval_accuracy_score'].rolling(ROLL_WINDOW, min_periods=1).mean()
    axes[0].plot(
        sub['step_sequence_number'], rolled,
        label=READ_ALGO_LABELS.get(algo, algo),
        color=palette_attn[i], lw=2,
    )
    axes[0].scatter(
        sub['step_sequence_number'],
        sub['retrieval_accuracy_score'],
        alpha=0.12, color=palette_attn[i], s=10,
    )

axes[0].set_xlabel('Step Sequence Number')
axes[0].set_ylabel('Retrieval Accuracy Score')
axes[0].set_title(f'Context Fidelity Over Steps (rolling avg={ROLL_WINDOW})')
axes[0].set_ylim(-0.05, 1.1)
axes[0].legend(fontsize=8)

# Right: mean retrieval accuracy vs mean compression ratio per read algorithm
summary_attn = attn_df.groupby('read_algorithm').agg(
    mean_accuracy=('retrieval_accuracy_score', 'mean'),
    mean_compression=('compression_ratio', 'mean'),
    mean_tokens=('input_tokens_used', 'mean'),
).reset_index()

scatter = axes[1].scatter(
    summary_attn['mean_compression'],
    summary_attn['mean_accuracy'],
    c=range(len(summary_attn)),
    cmap='tab10', s=200, zorder=3,
)
for _, row in summary_attn.iterrows():
    axes[1].annotate(
        READ_ALGO_LABELS.get(row['read_algorithm'], row['read_algorithm']),
        (row['mean_compression'], row['mean_accuracy']),
        textcoords='offset points', xytext=(6, 4), fontsize=8,
    )
axes[1].set_xlabel('Mean Compression Ratio')
axes[1].set_ylabel('Mean Retrieval Accuracy')
axes[1].set_title('Accuracy vs Compression: Algorithm Summary')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig8_attention_tracking.pdf', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig8_attention_tracking.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nMean retrieval accuracy by read algorithm:')
print(summary_attn[['read_algorithm', 'mean_accuracy', 'mean_compression', 'mean_tokens']].to_string(index=False))

## Table 1 — Statistical Significance (Wilcoxon Tests)

In [ ]:
wtest_files = sorted(Path('../results/experiment_c').glob('wilcoxon_*.csv'))
if wtest_files:
    all_tests = pd.concat([pd.read_csv(f) for f in wtest_files], ignore_index=True)
    display_cols = ['metric', 'comparison', 'n', 'p_value', 'significant_p05',
                    'effect_size', 'median_reduction']
    print(all_tests[display_cols].to_string(index=False))
else:
    print('No Wilcoxon test files found. Run experiments first.')

## Cost Summary

In [ ]:
total_cost = all_data['estimated_cost_usd'].sum()
total_iters = len(all_data)
print(f'Total iterations across all experiments: {total_iters}')
print(f'Total estimated API cost: ${total_cost:.4f}')
print(f'Average cost per iteration: ${total_cost/max(total_iters,1):.6f}')

print('\nToken breakdown by condition:')
print(all_data.groupby('condition')[[
    'simulator_input_tokens', 'simulator_output_tokens',
    'handoff_input_tokens', 'handoff_output_tokens',
    'input_tokens_used', 'output_tokens_used',
]].mean().round(1).to_string())